In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# Load dataset
df = pd.read_csv("shipping_cost_dataset.csv")

# Convert shipping mode to numbers
df["shipping_mode"] = df["shipping_mode"].map({
    "Sea":0,
    "Air":1,
    "Truck":2
})

# Features
X = df[[
    "distance_km",
    "fuel_price",
    "port_congestion",
    "container_size",
    "shipping_mode"
]]

# Target
y = df["shipping_cost"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor()

model.fit(X_train, y_train)

# Predict
pred = model.predict(X_test)

print("Model MAE:", mean_absolute_error(y_test, pred))

# Save model
joblib.dump(model, "shipping_model.pkl")

FileNotFoundError: [Errno 2] No such file or directory: 'shipping_cost_dataset.csv'

In [2]:
pip install pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install networkx

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 11.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import joblib
import networkx as nx
import random

# -------------------------------
# Load datasets
# -------------------------------

supplier_df = pd.read_csv(
    r"C:\Users\91939\OneDrive\Desktop\MLPROJECTS\tarrifs\supplier_trade_dataset.csv",
    encoding="latin1"
)

tariff_df = pd.read_csv(
    r"C:\Users\91939\OneDrive\Desktop\MLPROJECTS\tarrifs\tariff_dataset.csv",
    encoding="latin1"
)

shipping_df = pd.read_csv(
    r"C:\Users\91939\OneDrive\Desktop\MLPROJECTS\tarrifs\shipping_cost_dataset.csv",
    encoding="latin1"
)

# -------------------------------
# Load trained ML model
# -------------------------------

model = joblib.load("C:\\Users\\91939\\OneDrive\\Desktop\\MLPROJECTS\\tarrifs\\shipping_model.pkl")

# Convert shipping mode
shipping_df["shipping_mode"] = shipping_df["shipping_mode"].map({
    "Sea": 0,
    "Air": 1,
    "Truck": 2
})

# -------------------------------
# Show commodities
# -------------------------------

print("\nAvailable commodities:")
print(supplier_df["commodity"].unique())

commodity_choice = input("\nEnter commodity to import: ")

supplier_df = supplier_df[
    supplier_df["commodity"].str.lower() == commodity_choice.lower()
]

if supplier_df.empty:
    print("No suppliers found for this commodity")
    exit()

# -------------------------------
# Build global shipping network
# -------------------------------

G = nx.Graph()

G.add_node("Singapore")
G.add_node("Los Angeles")

countries = supplier_df["supplier_country"].unique()


for country in countries:

    if country == "Brazil":
        G.add_edge(country, "Los Angeles", distance=random.randint(8000,11000))

    else:
        G.add_edge(country, "Singapore", distance=random.randint(2000,8000))

# Main global shipping lane
G.add_edge("Singapore", "Los Angeles", distance=14000)

# -------------------------------
# Initialize results
# -------------------------------

best_supplier = None
best_route = None
best_time = None
lowest_cost = float("inf")

# -------------------------------
# Optimization loop
# -------------------------------

for i in range(len(supplier_df)):

    supplier = supplier_df.iloc[i]["supplier_country"]
    product_cost = supplier_df.iloc[i]["product_cost"]

    # Find best route
    route = nx.shortest_path(G, supplier, "Los Angeles", weight="distance")
    distance = nx.shortest_path_length(G, supplier, "Los Angeles", weight="distance")

    # Get shipping row
    shipping_match = shipping_df[
        shipping_df["supplier_country"] == supplier
    ]

    if shipping_match.empty:
        continue

    ship_row = shipping_match.iloc[0]

    # ML prediction features
    features = pd.DataFrame([{
        "distance_km": distance,
        "fuel_price": ship_row["fuel_price"],
        "port_congestion": ship_row["port_congestion"],
        "container_size": ship_row["container_size"],
        "shipping_mode": ship_row["shipping_mode"]
    }])

    predicted_shipping = model.predict(features)[0]

    # Get tariff
    tariff_match = tariff_df[
        tariff_df["supplier_country"] == supplier
    ]

    if tariff_match.empty:
        continue

    tariff_percent = tariff_match.iloc[0]["tariff_percent"]

    tariff_cost = product_cost * tariff_percent / 100

    insurance = 5
    handling = 8

    landed_cost = (
        product_cost
        + predicted_shipping
        + tariff_cost
        + insurance
        + handling
    )

    delivery_time = distance / 800

    print(f"Supplier: {supplier} | Landed Cost: {landed_cost:.2f}")

    if landed_cost < lowest_cost:
        lowest_cost = landed_cost
        best_supplier = supplier
        best_route = route
        best_time = delivery_time

# -------------------------------
# Final output
# -------------------------------

print("\n==========================")
print("BEST SUPPLY OPTION")
print("==========================")

print("Commodity:", commodity_choice)
print("Best Supplier:", best_supplier)
print("Best Route:", " → ".join(best_route))
print("Estimated Landed Cost:", round(lowest_cost,2))
print("Delivery Time:", round(best_time,1), "days")


Available commodities:
['Machinery' 'Steel' 'Textiles' 'Electronics']
Supplier: Thailand | Landed Cost: 248.89
Supplier: Indonesia | Landed Cost: 316.56
Supplier: Thailand | Landed Cost: 252.97
Supplier: Thailand | Landed Cost: 306.01
Supplier: Malaysia | Landed Cost: 297.02
Supplier: India | Landed Cost: 308.20
Supplier: China | Landed Cost: 175.27
Supplier: Malaysia | Landed Cost: 264.78
Supplier: Brazil | Landed Cost: 246.77
Supplier: Indonesia | Landed Cost: 168.72
Supplier: Malaysia | Landed Cost: 159.38
Supplier: Brazil | Landed Cost: 154.37
Supplier: Japan | Landed Cost: 211.39
Supplier: Brazil | Landed Cost: 147.77
Supplier: Malaysia | Landed Cost: 230.06
Supplier: Malaysia | Landed Cost: 272.22
Supplier: China | Landed Cost: 251.65
Supplier: Vietnam | Landed Cost: 204.06
Supplier: Vietnam | Landed Cost: 224.50
Supplier: Vietnam | Landed Cost: 296.04
Supplier: Thailand | Landed Cost: 183.61
Supplier: Thailand | Landed Cost: 233.93
Supplier: Thailand | Landed Cost: 212.17
Suppl

In [7]:
commodities = ["Steel", "Electronics", "Machinery", "Textiles"]

print("Available commodities:")

for i, c in enumerate(commodities, 1):
    print(f"{i}. {c}")

choice = int(input("Select commodity number: "))


print("Selected Commodity:", commodity_choice)

Available commodities:
1. Steel
2. Electronics
3. Machinery
4. Textiles
Selected Commodity: Steel


In [9]:
commodities = ["Steel", "Electronics", "Machinery", "Textiles"]

print("Available commodities:")

for i, c in enumerate(commodities, 1):
    print(f"{i}. {c}")

choice = int(input("Select commodity number: "))

commodity_choice = commodities[choice-1]

print("Selected Commodity:", commodity_choice)

print("BEST SUPPLY OPTION")


print(f"Commodity: {commodity_choice}")
print(f"Best Supplier: {best_supplier}")
print(f"Best Route: {' → '.join(best_route)}")
print(f"Estimated Landed Cost: {round(lowest_cost,2)}")
print(f"Delivery Time: {round(best_time,1)} days")

Available commodities:
1. Steel
2. Electronics
3. Machinery
4. Textiles
Selected Commodity: Machinery
BEST SUPPLY OPTION
Commodity: Machinery
Best Supplier: Brazil
Best Route: Brazil → Los Angeles
Estimated Landed Cost: 158.27
Delivery Time: 12.1 days
